# 파일과 예외

UTF-8 텍스트 파일을 안전하게 읽고, 예상한 오류만 처리하려면 어떻게 해야 할까요?

## 실행 환경 준비

Python 3.12 표준 라이브러리만 사용합니다. `TemporaryDirectory`가 실습 전용 폴더를 만들므로 개인 파일을 변경하지 않습니다.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="ch02_05_")
training_dir = Path(workspace.name)
orders_path = training_dir / "orders.txt"
orders_path.write_text("연필,3\n노트,2\n", encoding="utf-8")
print("준비한 파일:", orders_path.name)

준비한 파일: orders.txt


## 기본 실행

`read_text()`에 UTF-8 인코딩을 명시해 한글을 읽습니다. `splitlines()`는 줄 단위 목록을 반환합니다.

In [2]:
def read_items(path: Path) -> list[str]:
    text = path.read_text(encoding="utf-8")
    return text.splitlines()

items = read_items(orders_path)
print(items)
print("주문 줄 수:", len(items))

['연필,3', '노트,2']
주문 줄 수: 2


## 중간 결과 관찰

읽은 각 줄을 쉼표에서 나누어 상품명과 수량을 확인합니다.

In [3]:
def parse_order(line: str) -> tuple[str, int]:
    name, count_text = line.split(",")
    return name, int(count_text)

parsed = [parse_order(line) for line in items]
print(parsed)
assert parsed[0] == ("연필", 3)

[('연필', 3), ('노트', 2)]


## 값 하나를 바꿔 비교

같은 인코딩과 처리 함수를 유지하고 파일 내용의 수량만 바꿉니다.

In [4]:
orders_path.write_text("연필,5\n노트,2\n", encoding="utf-8")
changed = [parse_order(line) for line in read_items(orders_path)]
print("변경 전 첫 수량:", parsed[0][1])
print("변경 후 첫 수량:", changed[0][1])
assert changed[0][1] == 5

변경 전 첫 수량: 3
변경 후 첫 수량: 5


## 실패 재현

존재하지 않는 파일을 읽으면 `FileNotFoundError`가 발생합니다. 오류의 종류와 파일 이름을 확인합니다.

In [5]:
missing_path = training_dir / "missing.txt"
try:
    read_items(missing_path)
except FileNotFoundError as error:
    print(type(error).__name__, missing_path.name)
else:
    raise AssertionError("FileNotFoundError가 발생해야 합니다.")

FileNotFoundError missing.txt


## 수정

파일이 없을 때만 빈 목록을 반환하는 함수를 만듭니다. 다른 오류는 감추지 않으므로 원인을 계속 확인할 수 있습니다.

In [6]:
def read_items_or_empty(path: Path) -> list[str]:
    try:
        return read_items(path)
    except FileNotFoundError:
        return []

result = read_items_or_empty(missing_path)
print("누락 파일 결과:", result)
assert result == []

누락 파일 결과: []
